**LSTM FINANCIAL TIME SERIES MODEL ON NIFTY50 AND S&P500**

In [184]:
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns

**DOWNLOADING DATA**

In [185]:
nifty = yf.download('^NSEI', start='2018-01-01', end='2024-01-01')
sp500 = yf.download('^GSPC', start='2018-01-01', end='2024-01-01')

/tmp/ipykernel_2632/4183048594.py:1: FutureWarning: YF.download() has changed argument auto_adjust default to True
  nifty = yf.download('^NSEI', start='2018-01-01', end='2024-01-01')
[*********************100%***********************]  1 of 1 completed
/tmp/ipykernel_2632/4183048594.py:2: FutureWarning: YF.download() has changed argument auto_adjust default to True
  sp500 = yf.download('^GSPC', start='2018-01-01', end='2024-01-01')
[*********************100%***********************]  1 of 1 completed


In [186]:
print(nifty.head())
print(sp500.head())

Price              Close          High           Low          Open  Volume
Ticker             ^NSEI         ^NSEI         ^NSEI         ^NSEI   ^NSEI
Date                                                                      
2018-01-02  10442.200195  10495.200195  10404.650391  10477.549805  153400
2018-01-03  10443.200195  10503.599609  10429.549805  10482.650391  167300
2018-01-04  10504.799805  10513.000000  10441.450195  10469.400391  174900
2018-01-05  10558.849609  10566.099609  10520.099609  10534.250000  180900
2018-01-08  10623.599609  10631.200195  10588.549805  10591.700195  169000
Price             Close         High          Low         Open      Volume
Ticker            ^GSPC        ^GSPC        ^GSPC        ^GSPC       ^GSPC
Date                                                                      
2018-01-02  2695.810059  2695.889893  2682.360107  2683.729980  3397430000
2018-01-03  2713.060059  2714.370117  2697.770020  2697.850098  3544030000
2018-01-04  2723.989990  

In [187]:
print(nifty.shape)
print(sp500.shape)

(1477, 5)
(1509, 5)


In [188]:
nifty.isnull().sum().sum()
sp500.isnull().sum().sum()

np.int64(0)

**FEATURE ENGINEERING**

In [189]:
nifty['returns'] = nifty['Close']['^NSEI'].pct_change(fill_method=None)

In [190]:
volume_nifty = nifty['Volume']['^NSEI']

In [191]:
nifty['Volume Ratio'] = (volume_nifty/volume_nifty.rolling(10).mean()).shift(1)

In [192]:
nifty['Rolling Returns'] = nifty['returns'].rolling(20).mean().shift(1)

In [193]:
delta = nifty['Close']['^NSEI'].diff()
gain = delta.clip(lower=0).rolling(14).mean()
loss = (-delta.clip(upper=0)).rolling(14).mean()
RSI = 100 - (100 / (1 + gain/loss))
nifty['RSI'] = RSI.shift(1)

In [194]:
sp500['SP500 returns'] = sp500['Close']['^GSPC'].pct_change(fill_method=None)

In [195]:
sp500_return_dataframe = pd.DataFrame({
    'Date': sp500.index,
    'SP500 returns': sp500['SP500 returns']
})
sp500_return_dataframe.set_index('Date', inplace=True)

In [196]:
nifty_clean = pd.DataFrame({
    "returns": nifty['returns'],
    "Volume Ratio": nifty['Volume Ratio'],
    "Rolling Returns": nifty['Rolling Returns'],
    "RSI": nifty['RSI']
})

In [198]:
nifty_clean = nifty_clean.join(sp500_return_dataframe,how="inner")

,returns,Volume Ratio,Rolling Returns,RSI,SP500 returns
Date,,,,,
2018-01-02,NaN,NaN,NaN,NaN,NaN
2018-01-03,0.000096,NaN,NaN,NaN,0.006399
2018-01-04,0.005899,NaN,NaN,NaN,0.004029
2018-01-05,0.005145,NaN,NaN,NaN,0.007034
2018-01-08,0.006132,NaN,NaN,NaN,0.001662
...,...,...,...,...,...
2023-12-22,0.004439,0.926668,0.003547,75.659666,0.001660
2023-12-26,0.004307,0.953162,0.003794,70.719263,0.004232
2023-12-27,0.009953,0.733255,0.004028,69.251046,0.001430


In [202]:
nifty_clean['target'] = (nifty_clean['returns'] > 0).astype(int).shift(-1)
nifty_clean.dropna(inplace=True)
nifty_clean

,returns,Volume Ratio,Rolling Returns,RSI,SP500 returns,target
Date,,,,,,
2018-02-01,-0.000979,0.939338,0.002741,77.626477,-0.000648,0.0
2018-02-02,-0.023264,1.146258,0.002687,75.840934,-0.021209,0.0
2018-02-05,-0.008740,1.070023,0.001229,54.248522,-0.040979,0.0
2018-02-06,-0.015778,0.897738,0.000535,46.124430,0.017441,0.0
2018-02-07,-0.002053,0.983613,-0.000561,40.765426,-0.005002,1.0
...,...,...,...,...,...,...
2023-12-21,0.004960,1.216203,0.003371,76.031608,0.010301,1.0
2023-12-22,0.004439,0.926668,0.003547,75.659666,0.001660,1.0
2023-12-26,0.004307,0.953162,0.003794,70.719263,0.004232,1.0


In [203]:
nifty_clean.shape

(1414, 6)

In [204]:
nifty_clean.columns.tolist()

['returns',
 'Volume Ratio',
 'Rolling Returns',
 'RSI',
 'SP500 returns',
 'target']